# Exploitation Zone — Feature derivation pipeline

Same flow as [`exploitation_zone_processing.py`](../exploitation_zone_processing.py):

1. **Spark batch (Docker):** trusted audio → spectral features → `metadata/spark_pending_workset.json`
2. **Host Python:** MFCCs + `harmonic_energy_ratio` per row
3. Append exploitation-zone CSV + Parquet
4. Sync Parquet → Delta Lake

Run **trusted zone** first. Rows already in exploitation metadata are skipped (Spark anti-join).

## Paths and environment

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
EXPLOITATION_ZONE_DIR = _root / "exploitation_zone"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(EXPLOITATION_ZONE_DIR) not in sys.path:
    sys.path.insert(1, str(EXPLOITATION_ZONE_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("EXPLOITATION_ZONE_BUCKET =", os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone"))

## Imports

In [ ]:
import importlib

import exploitation_zone_processing as ez
importlib.reload(ez)

from shared.minio_helpers import (
    MINIO_ACCESS_KEY,
    MINIO_SECRET_KEY,
    MINIO_ENDPOINT,
    create_minio_client,
    is_unreachable_minio,
    append_rows_to_csv,
    update_parquet,
)
from shared.sync_delta import sync_observations_to_delta
from spark_exploitation_zone import load_spark_pending_workset, run_spark_exploitation_zone_subprocess

## [1/4] MinIO client and exploitation bucket

In [ ]:
if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise RuntimeError("Set MINIO_ACCESS_KEY and MINIO_SECRET_KEY")

client = create_minio_client()
try:
    ez.ensure_exploitation_bucket(client)
except Exception as e:
    if is_unreachable_minio(e):
        raise RuntimeError(
            f"MinIO at {MINIO_ENDPOINT!r} is unreachable while preparing {ez.EXPLOITATION_BUCKET!r}."
        ) from e
    raise RuntimeError(f"Exploitation bucket setup failed: {e}") from e

## [2/4] Spark batch — trusted audio → spectral features

In [ ]:
print("\n[1/3] Spark batch (Docker): trusted audio → spectral features...")
try:
    run_spark_exploitation_zone_subprocess(project_root=PROJECT_ROOT)
    pending = load_spark_pending_workset(client, bucket=ez.EXPLOITATION_BUCKET)
except Exception as e:
    raise RuntimeError("Exploitation-zone Spark batch failed — " + str(e)) from e

if not pending:
    print(
        "  No pending rows — trusted metadata empty or all UUIDs already in exploitation-zone."
    )
else:
    print(f"  Pending rows after Spark batch: {len(pending)}")
    print(f"\n[2/3] Python features ({', '.join(ez.PYTHON_AUDIO_FEATURE_COLUMNS)}):")

## [3/4] Python features (MFCC + harmonic ratio)

In [ ]:
import time

rows = []
if pending:
    for i, rec in enumerate(pending):
        uid = str(rec.get("uuid", "?"))[:8]
        print(f"\n  [{i+1}/{len(pending)}] uuid={uid}...")
        t0 = time.perf_counter()
        row = ez.enrich_python_features(client, rec)
        if row:
            row["feature_processing_time(seconds)"] = time.perf_counter() - t0
            rows.append(row)
            print(
                f"    Done — harmonic_ratio={row.get('harmonic_energy_ratio', 0):.3f}  "
                f"({row['feature_processing_time(seconds)']:.1f}s)"
            )

if not rows:
    print("\n  No records enriched.")
else:
    print(f"\n  Enriched {len(rows)} record(s).")

## [4/4] Write metadata and sync Delta

In [ ]:
if rows:
    fieldnames = list(ez.EXPLOITATION_METADATA_FIELDS)
    if "feature_processing_time(seconds)" not in fieldnames:
        fieldnames.append("feature_processing_time(seconds)")
    print(f"\n[3/3] Writing exploitation metadata ({len(rows)} rows)...")
    append_rows_to_csv(
        client,
        ez.EXPLOITATION_BUCKET,
        rows,
        fieldnames,
        key=ez.METADATA_KEY,
    )
    print(f"  Updated: {ez.METADATA_KEY}")
    update_parquet(
        client,
        ez.EXPLOITATION_BUCKET,
        rows,
        key=ez.METADATA_KEY.replace(".csv", ".parquet"),
    )

    print("\n[4/4] Syncing Parquet → Delta Lake...")
    sync_observations_to_delta(client, ez.EXPLOITATION_BUCKET, zone_label="exploitation")

    print("\n  Exploitation-zone processing complete.")
    print(f"   Processed: {len(rows)} records")
    print(f"   Bucket: {ez.EXPLOITATION_BUCKET}")
    print(f"   Metadata: {ez.METADATA_KEY}")